In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

# 1. โหลดข้อมูล
train_df = pd.read_csv("../cleaned_data/train_data.csv")
test_df = pd.read_csv("../cleaned_data/test_data.csv")


# 2. Feature Engineering
def engineer_features(df):
    df = df.copy()
    df['is_senior'] = (df['age'] >= 60).astype(int)
    df['has_free_cash'] = ((df['balance'] > 0) & (df['housing'] == 'no') & (df['loan'] == 'no')).astype(int)
    df['total_contacts'] = df['campaign'] + df['previous']
    df['is_balance_negative'] = (df['balance'] < 0).astype(int)
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

X_train = train_fe.drop(columns=['y'])
y_train = train_fe['y']
X_test = test_fe.drop(columns=['y'])
y_test = test_fe['y']

# 3. One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 4. Feature Scaling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded), columns=X_train_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_encoded), columns=X_test_encoded.columns)

# 5. Feature Selection (เก็บฟีเจอร์ที่สำคัญระดับค่ามัธยฐานขึ้นไป = 22 ฟีเจอร์)
selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1), 
    threshold='median'
)
selector.fit(X_train_scaled, y_train)

selected_features = X_train_scaled.columns[selector.get_support()]
X_train_selected = X_train_scaled[selected_features]
X_test_selected = X_test_scaled[selected_features]

# 6. Hyperparameter Tuning & Cross-Validation
param_dist = {
    'n_estimators': [200, 300, 400],
    'max_depth': [12, 15, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 5]
}

rf_base = RandomForestClassifier(random_state=0, class_weight='balanced')

random_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=param_dist, 
    n_iter=15, 
    cv=5, # เพิ่มเป็น 5-Fold เพื่อความเสถียรสูงสุด
    scoring='f1', 
    random_state=0, 
    n_jobs=-1
)

print("กำลังเทรน Random Forest Final...")
random_search.fit(X_train_selected, y_train)
best_rf_model = random_search.best_estimator_

# 7. ประเมินผล (ปรับ Threshold เป็น 0.48 เพื่อความสมดุลระหว่าง Precision และ Recall)
y_probs = best_rf_model.predict_proba(X_test_selected)[:, 1]
y_pred = (y_probs >= 0.48).astype(int)

print("\n=== RANDOM FOREST FINAL REPORT ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_rf_model, "../models/rf_model.pkl")
X_test_selected.to_csv("../cleaned_data/X_test_selected_rf.csv", index=False)
print("✅ บันทึก Random Forest และ X_test (22 features) เรียบร้อย!")

กำลังเทรน Random Forest Final...

=== RANDOM FOREST FINAL REPORT ===
Accuracy: 0.8230119226762357

Confusion Matrix:
 [[6550 1085]
 [ 444  560]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.86      0.90      7635
           1       0.34      0.56      0.42      1004

    accuracy                           0.82      8639
   macro avg       0.64      0.71      0.66      8639
weighted avg       0.87      0.82      0.84      8639

✅ บันทึก Random Forest และ X_test (22 features) เรียบร้อย!
